# Penjelasan Model Agnostic

Model kompleks seperti random forest sulit ditafsirkan langsung. Metode *model-agnostic* menjelaskan prediksinya tanpa melihat isi model.

## Tujuan

- Menghitung permutation feature importance.
- Menggambar partial dependence plot.
- Membangun surrogate model global.
- Membangun surrogate model lokal untuk satu prediksi.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn import metrics
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import PartialDependenceDisplay, permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, export_text

RANDOM_STATE = 123


## Memuat data

Data berisi profil karyawan. Targetnya menandai apakah seorang karyawan sedang mencari pekerjaan baru.


In [ ]:
DATA_URL = (
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBM-ML201EN-SkillsNetwork/labs/module_4/datasets/hr_new_job_processed.csv"
)
job_data = pd.read_csv(DATA_URL)
job_data.describe()


In [ ]:
features = job_data.loc[:, job_data.columns != "target"]
target = job_data["target"]

X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.2, stratify=target, random_state=12
)


## Model black-box

Random forest dipakai sebagai model yang akan dijelaskan. Model ini akurat, tetapi aturannya terlalu banyak untuk dibaca manusia.


In [ ]:
black_box = RandomForestClassifier(
    random_state=RANDOM_STATE, max_depth=25, max_features=10,
    n_estimators=100, bootstrap=True, n_jobs=-1,
).fit(X_train, y_train)


In [ ]:
black_box_predictions = black_box.predict(X_test)
metrics.roc_auc_score(y_test, black_box_predictions)


## Permutation feature importance

Nilai sebuah fitur diacak, lalu diukur seberapa jauh akurasi turun. Penurunan besar berarti fitur itu penting.


In [ ]:
importance = permutation_importance(
    estimator=black_box, X=X_train, y=y_train,
    n_repeats=5, random_state=RANDOM_STATE, n_jobs=2,
)
importance.importances.shape


In [ ]:
def visualize_feature_importance(result):
    order = result.importances_mean.argsort()
    figure, axis = plt.subplots(figsize=(14, 8))
    axis.boxplot(
        result.importances[order].T,
        vert=False,
        tick_labels=X_train.columns[order],
    )
    axis.set_title("Permutation Importances (train set)")
    plt.tight_layout()
    plt.show()


visualize_feature_importance(importance)


Fitur yang paling penting biasanya berkaitan dengan indeks pembangunan kota dan pengalaman kerja.


## Latihan: ulangi dengan `n_repeats=10`


In [ ]:
repeated_importance = permutation_importance(
    estimator=black_box, X=X_train, y=y_train,
    n_repeats=10, random_state=RANDOM_STATE, n_jobs=2,
)
visualize_feature_importance(repeated_importance)


## Partial dependence plot

PDP menunjukkan bagaimana prediksi rata-rata berubah ketika satu fitur divariasikan, sementara fitur lain dibiarkan.


In [ ]:
important_features = ["city_development_index", "experience"]
PartialDependenceDisplay.from_estimator(
    black_box, X_train, features=important_features, random_state=RANDOM_STATE
)
plt.tight_layout()
plt.show()


## Latihan: PDP untuk fitur lain


In [ ]:
PartialDependenceDisplay.from_estimator(
    black_box, X_train,
    features=["company_size", "education_level", "training_hours"],
    random_state=RANDOM_STATE,
)
plt.tight_layout()
plt.show()


## Surrogate model global

Surrogate model adalah model sederhana yang dilatih untuk meniru keluaran model black-box. Semakin tinggi akurasinya, semakin layak penjelasannya dipercaya.


In [ ]:
scaler = StandardScaler()
X_test_scaled = scaler.fit_transform(X_test)

logistic_surrogate = LogisticRegression(
    max_iter=1000, random_state=RANDOM_STATE, penalty="l1", solver="liblinear"
).fit(X_test_scaled, black_box_predictions)

surrogate_predictions = logistic_surrogate.predict(X_test_scaled)
metrics.accuracy_score(black_box_predictions, surrogate_predictions)


In [ ]:
coefficients = {
    feature: coefficient
    for coefficient, feature in zip(logistic_surrogate.coef_[0, :], X_test.columns)
    if abs(coefficient) >= 0.01
}
coefficients = dict(sorted(coefficients.items(), key=lambda item: item[1]))

figure, axis = plt.subplots(figsize=(10, 6))
axis.barh(list(coefficients.keys()), list(coefficients.values()),
          color=["r" if value <= 0 else "g" for value in coefficients.values()])
axis.set_xlabel("Feature Coefficients")
plt.tight_layout()
plt.show()


Fitur dengan koefisien negatif menurunkan peluang karyawan berganti pekerjaan, sedangkan koefisien positif menaikkannya.


## Latihan: surrogate model pohon keputusan


In [ ]:
tree_surrogate = DecisionTreeClassifier(max_depth=4, random_state=RANDOM_STATE)
tree_surrogate.fit(X_test, black_box_predictions)
tree_predictions = tree_surrogate.predict(X_test)

metrics.accuracy_score(black_box_predictions, tree_predictions)


In [ ]:
print(export_text(tree_surrogate, feature_names=list(X_train.columns)))


## Surrogate model lokal

Penjelasan global bisa meleset untuk satu individu. Surrogate lokal dilatih hanya pada sampel di sekitar satu observasi, sehingga menjelaskan prediksi observasi tersebut.

Sampel tetangga dibuat dengan menambahkan derau kecil pada fitur, lalu diberi label oleh model black-box.


In [ ]:
instance_index = 19
selected_instance = X_test.iloc[[instance_index]]
selected_instance


In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
neighbor_count = 500

neighbors = selected_instance.values + rng.normal(
    scale=0.05 * X_train.std().values, size=(neighbor_count, X_train.shape[1])
)
neighbors = pd.DataFrame(neighbors, columns=X_train.columns)
neighbor_labels = black_box.predict(neighbors)

local_surrogate = DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE)
local_surrogate.fit(neighbors, neighbor_labels)

local_importance = pd.Series(
    local_surrogate.feature_importances_, index=X_train.columns
).sort_values(ascending=False)
local_importance.head(6)


In [ ]:
local_importance.head(6).sort_values().plot.barh(figsize=(8, 4))
plt.xlabel("Local importance")
plt.title("Fitur penentu untuk satu karyawan")
plt.tight_layout()
plt.show()


## Ringkasan

Permutation importance dan PDP menjelaskan perilaku model secara umum. Surrogate global memberi gambaran sederhana yang bisa dibaca manusia, sedangkan surrogate lokal menjelaskan satu prediksi tertentu. Semua metode ini hanya mendekati perilaku model, bukan membuktikan hubungan sebab akibat.
